In [24]:

import torch

class DLGN(torch.nn.Module):
    def __init__(self,input_dim,hidden_dim,num_layers, beta = 1):

        super(DLGN,self).__init__()
        self.beta = beta
        self.input_dim = input_dim
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers

        self.hiddens = torch.nn.ParameterList([torch.nn.Parameter(torch.randn(input_dim,hidden_dim),requires_grad=True)])
        self.hiddens.extend([torch.nn.Parameter(torch.randn(hidden_dim,hidden_dim),requires_grad=True) for _ in range(num_layers - 1)])
        self.Us = torch.nn.ParameterList([torch.nn.Parameter(torch.randn(hidden_dim,1),requires_grad=True)])
        ulast = torch.nn.Parameter(torch.randn(hidden_dim,1))
        self.Us.extend([torch.nn.Parameter(torch.randn(hidden_dim,hidden_dim),requires_grad=True) for _ in range(num_layers - 1)])
        self.Us.extend([ulast])
        self.biases = torch.nn.ParameterList([torch.nn.Parameter(torch.randn(hidden_dim)) for _ in range(num_layers)])

    def forward(self,x):

        self.V = [self.hiddens[0]]
        h = torch.nn.Sigmoid(self.beta*(self.V[0]@x + self.biases[0])) * self.Us[0]

        for i in range(1,self.num_layers):
            self.V.append(self.V[-1]@self.hiddens[i])
            h = torch.nn.Sigmoid(self.beta*(self.V[i]@x + self.biases[i]))*(self.Us[i]@h)
        return torch.dot(h,self.Us[-1])



        
        


In [ ]:
# def data_gen_decision_tree(num_data=1000, dim=2, seed=0, w_list=None, b_list=None,vals=None, num_levels=2,): 
#     torch.manual_seed(seed)
#     data_x = torch.randn(10,2)
#     data_x /= torch.linalg.norm(data_x,dim=1,keepdim=True)

#     if w_list is None:
#         w_list = torch.randn()




In [1]:

def data_gen_decision_tree(num_data=1000, dim=2, seed=0, w_list=None, b_list=None,vals=None, num_levels=2):        
    #set_npseed(seed=seed)
    import numpy as np

    

    # Construct a complete decision tree with 2**num_levels-1 internal nodes,
    # e.g. num_levels=2 means there are 3 internal nodes.
    # w_list, b_list is a list of size equal to num_internal_nodes
    # vals is a list of size equal to num_leaf_nodes, with values +1 or 0
    num_internal_nodes = 2**num_levels - 1
    num_leaf_nodes = 2**num_levels
    stats = np.zeros(num_internal_nodes+num_leaf_nodes) #stores the num of datapoints at each node so at 0(root) all data points will be present

    if vals is None: #when val i.e., labels are not provided make the labels dynamically
        vals = np.arange(0,num_internal_nodes+num_leaf_nodes,1,dtype=np.int32)%2 #assign 0 or 1 label to the node based on whether its numbering is even or odd
        vals[:num_internal_nodes] = -99 #we put -99 to the internal nodes as only the values of leaf nodes are counted

    if w_list is None: #if the w values of the nodes (hyperplane eqn) are not provided then generate dynamically
        w_list = np.random.standard_normal((num_internal_nodes, dim))
        w_list = w_list/np.linalg.norm(w_list, axis=1)[:, None] #unit norm w vects
        b_list = np.zeros((num_internal_nodes))

    

#     data_x = np.random.random_sample((num_data, dim))*2 - 1. #generate the datas in range -1 to +1
#     relevant_stats = data_x @ w_list.T + b_list #stores the x.wT+b value of each nodes for all data points(num_data x num_nodes) to check if > 0 i.e will follow right sub tree route or <0 and will follow left sub tree route
#     curr_index = np.zeros(shape=(num_data), dtype=int) #stores the curr index for each data point from root to leaf. So initially a datapoint starts from root but then it can go to right or left if it goes to right its curr index will become 2 from 0 else 1 from 0 then in next iteration from say 2 it goes to right then it will become 6

    data_x = np.random.standard_normal((num_data, dim))
    data_x /= np.sqrt(np.sum(data_x**2, axis=1, keepdims=True))
    relevant_stats = data_x @ w_list.T + b_list
    curr_index = np.zeros(shape=(num_data), dtype=int)
    
    for level in range(num_levels):
        nodes_curr_level=list(range(2**level - 1,2**(level+1)-1  ))
        for el in nodes_curr_level:
#             b_list[el]=-1*np.median(relevant_stats[curr_index==el,el])
            relevant_stats[:,el] += b_list[el]
        decision_variable = np.choose(curr_index, relevant_stats.T) #based on the curr index will choose the corresponding node value of the datapoint

        # Go down and right if wx+b>0 down and left otherwise.
        # i.e. 0 -> 1 if w[0]x+b[0]<0 and 0->2 otherwise
        curr_index = (curr_index+1)*2 - (1-(decision_variable > 0)) #update curr index based on the desc_variable
        

    bound_dist = np.min(np.abs(relevant_stats), axis=1) #finds the abs value of the minm node value of a datapoint. If some node value of a datapoint is 0 then that data point exactly passes through a hyperplane and we remove all such datapoints
    thres = threshold
    labels = vals[curr_index] #finally labels for each datapoint is assigned after traversing the whole tree

    data_x_pruned = data_x[bound_dist>thres] #to distingush the hyperplanes seperately for 0 1 labels (classification)
    #removes all the datapoints that passes through a node hyperplane
    labels_pruned = labels[bound_dist>thres]
    relevant_stats = np.sign(data_x_pruned @ w_list.T + b_list) #storing only +1 or -1 for a particular node if it is active or not
    nodes_active = np.zeros((len(data_x_pruned),  num_internal_nodes+num_leaf_nodes), dtype=np.int32) #stores node actv or not for a data

    for node in range(num_internal_nodes+num_leaf_nodes):
        if node==0:
            stats[node]=len(relevant_stats) #for root node all datapoints are present
            nodes_active[:,0]=1 #root node all data points active status is +1
            continue
        parent = (node-1)//2
        nodes_active[:,node]=nodes_active[:,parent]
        right_child = node-(parent*2)-1 # 0 means left, 1 means right 1 has children 3,4
        #finds if it is a right child or left of the parent
        if right_child==1:
            nodes_active[:,node] *= relevant_stats[:,parent]>0 #if parent node val was >0 then this right child of parent is active
        if right_child==0:
            nodes_active[:,node] *= relevant_stats[:,parent]<0 #else left is active
        stats = nodes_active.sum(axis=0) #updates the status i.e., no of datapoints active in that node (root has all active then gradually divided in left right)
    return ((data_x_pruned, labels_pruned), (w_list, b_list, vals), stats)

In [4]:
import numpy as np
threshold = 0

(X,y),params,stats = data_gen_decision_tree()

np.save('synthetic_features.npy',X)
np.save('synthetic_labels.npy',y)


In [9]:
from torch.utils.data import Dataset, DataLoader
import torch

class decision_tree_dataset(Dataset):
    def __init__(self,X,y):
        self.X = torch.from_numpy(X)
        self.y = torch.from_numpy(y)
    def __len__(self):
        return len(self.X)
    def __getitem__(self,id):
        return self.X[id],self.y[id]




In [ ]:
from torch.utils.data import random_split

dataset = decision_tree_dataset(data[0],data[1])


generator = torch.Generator().manual_seed(42)

train, test = random_split(dataset,[0.3,0.7],generator = generator)







(tensor([-0.9677, -0.2521], dtype=torch.float64), tensor(0, dtype=torch.int32))

torch.Size([2, 20])
torch.Size([20, 20])
torch.Size([20, 20])
torch.Size([20, 1])
torch.Size([20, 20])
torch.Size([20, 20])
torch.Size([20, 1])
torch.Size([20])
torch.Size([20])
torch.Size([20])


In [ ]:
from torch.utils.data import random_split
from torch.nn import CrossEntropyLoss
from torch.optim import SGD

def train(model, train_loader, epochs = 100, device = torch.device('cuda' if torch.cuda.is_available() else 'cpu'),lr = 0.01):
    loss = 0
    loss_fn = CrossEntropyLoss()

    losses = []
    optimizer = SGD(model.parameters(),lr = lr)
    
    for epoch in epochs:
        model.train(True)

        for features,label in train_loader:
            optimizer.zero_grad()
            outputs = model(features)
            loss = 




